# Step 3: RAG Pipeline for BHC Generation
# ICU Patient Summary Generation — Group 1

**Goal:** Replace naive concatenation + truncation with retrieval-augmented generation.
Instead of feeding all notes blindly (and losing content for 42/100 patients),
we chunk → embed → retrieve the most relevant pieces → generate.

**Same model** (Mistral 7B) and **same generation settings** as baseline for fair comparison.
The only variable is *what goes into the context window*.

**Environment:** Google Colab Pro, A100 GPU, High RAM

In [1]:
# Mount Google Drive to access our processed data files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Verify we can see our data files from Steps 1 and 2
import os

DATA_DIR = "/content/drive/MyDrive/BMI702(Project)/Data"

print("Files in data directory:")
for f in sorted(os.listdir(DATA_DIR)):
    print(f"  {f}")

Files in data directory:
  baseline_eval_results.parquet
  baseline_results.parquet
  human_bhcs.parquet
  notes.parquet
  propositions_with_gt.parquet


In [5]:
# Load the processed data from our exploration notebook + baseline results
import pandas as pd
import numpy as np

notes = pd.read_parquet(f"{DATA_DIR}/notes.parquet")
human_bhcs = pd.read_parquet(f"{DATA_DIR}/human_bhcs.parquet")
baseline_results = pd.read_parquet(f"{DATA_DIR}/baseline_results.parquet")
baseline_eval = pd.read_parquet(f"{DATA_DIR}/baseline_eval_results.parquet")

print(f"Notes: {notes.shape[0]} rows")
print(f"BHC targets: {human_bhcs.shape[0]} patients")
print(f"Baseline results: {baseline_results.shape[0]} patients")
print(f"Baseline eval: {baseline_eval.shape[0]} patients")
print(f"\nBaseline ROUGE-L F1: {baseline_eval['rougeL_f'].mean():.3f}")

Notes: 4787 rows
BHC targets: 100 patients
Baseline results: 100 patients
Baseline eval: 100 patients

Baseline ROUGE-L F1: 0.165


In [6]:
# Install packages (same as baseline + new ones for RAG)
# sentence-transformers: for the embedding model (converts text → vectors)
!pip install -q transformers accelerate bitsandbytes torch sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.3 MB/s eta 0:00:00


In [7]:
# Log in to Hugging Face
from huggingface_hub import login
login()

In [8]:
# Load the SAME model with the SAME settings as baseline
# The only thing changing between baseline and RAG is WHAT goes into the context window
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.3"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print(f"Model loaded: {model_name}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded: mistralai/Mistral-7B-Instruct-v0.3
GPU memory used: 4.1 GB


In [9]:
# Load BGE-M3: a biomedical-capable embedding model
# This is what the VeriFact paper used for their RAG pipeline
# It converts text into 1024-dimensional vectors where similar medical concepts end up close together in vector space
from sentence_transformers import SentenceTransformer

embed_model_name = "BAAI/bge-m3"
embed_model = SentenceTransformer(embed_model_name)

# Quick sanity check — embed a short clinical phrase
test_embedding = embed_model.encode(["Patient admitted with chest pain"])
print(f"Embedding model: {embed_model_name}")
print(f"Embedding dimension: {test_embedding.shape[1]}")
print(f"First 5 values: {test_embedding[0][:5]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding model: BAAI/bge-m3
Embedding dimension: 1024
First 5 values: [ 0.03787659 -0.01465984 -0.04063635 -0.00986062 -0.01515621]


## 2. Chunking Clinical Notes

Break each note into ~300-token chunks with metadata preserved.
Each chunk knows which patient it belongs to, what type of note it came from,
and when it was written, so retrieved context stays interpretable.

**Why ~300 tokens?** Small enough to be focused on one topic, large enough
to preserve clinical context. A typical paragraph in a physician note.

In [10]:
# Chunking strategy: split each note into ~300-token pieces
# We preserve metadata (patient ID, note category, date) with each chunk so the LLM knows where retrieved content came from

def chunk_note(row, tokenizer, target_tokens=300, overlap_tokens=50):
    """
    Split a single clinical note into overlapping chunks.

    Args:
        row: A row from our notes dataframe (has TEXT, SUBJECT_ID, CATEGORY, CHARTDATE)
        tokenizer: The Mistral tokenizer (so token counts match what the LLM sees)
        target_tokens: Approximate size of each chunk in tokens
        overlap_tokens: How many tokens to repeat between chunks (preserves context)

    Returns:
        List of chunk dictionaries with text and metadata
    """
    text = row['TEXT']
    tokens = tokenizer.encode(text)

    # If the note is short enough, keep it as one chunk
    if len(tokens) <= target_tokens:
        return [{
            'subject_id': row['SUBJECT_ID'],
            'category': row['CATEGORY'],
            'chartdate': str(row['CHARTDATE']),
            'chunk_text': text,
            'chunk_tokens': len(tokens),
            'note_row_id': row.name  # track which original note this came from
        }]

    # Otherwise, split into overlapping windows
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + target_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)

        chunks.append({
            'subject_id': row['SUBJECT_ID'],
            'category': row['CATEGORY'],
            'chartdate': str(row['CHARTDATE']),
            'chunk_text': chunk_text,
            'chunk_tokens': len(chunk_tokens),
            'note_row_id': row.name
        })

        # Move forward by (target - overlap) so chunks overlap slightly
        # This prevents important info from being split across chunk boundaries
        start += target_tokens - overlap_tokens

    return chunks

In [11]:
import time

print("Chunking all 4,787 clinical notes...")
start = time.time()

all_chunks = []
for _, row in notes.iterrows():
    all_chunks.extend(chunk_note(row, tokenizer))

chunks_df = pd.DataFrame(all_chunks)
elapsed = time.time() - start

print(f"Done in {elapsed:.1f}s")
print(f"\nChunking summary:")
print(f"  Original notes: {len(notes)}")
print(f"  Total chunks: {len(chunks_df)}")
print(f"  Avg chunks per note: {len(chunks_df) / len(notes):.1f}")
print(f"  Chunk size: mean {chunks_df['chunk_tokens'].mean():.0f}, "
      f"median {chunks_df['chunk_tokens'].median():.0f} tokens")

# How many chunks per patient?
per_patient = chunks_df.groupby('subject_id').size()
print(f"\nChunks per patient:")
print(f"  Mean: {per_patient.mean():.0f}")
print(f"  Median: {per_patient.median():.0f}")
print(f"  Range: {per_patient.min()}–{per_patient.max()}")

Chunking all 4,787 clinical notes...
Done in 13.0s

Chunking summary:
  Original notes: 4787
  Total chunks: 23663
  Avg chunks per note: 4.9
  Chunk size: mean 266, median 300 tokens

Chunks per patient:
  Mean: 237
  Median: 110
  Range: 33–1561


## 3. Embed Chunks and Build FAISS Indices

Convert each chunk into a 1024-dim vector using BGE-M3, then build
a per-patient FAISS index for fast similarity search. Each patient gets their own isolated search index.

In [12]:
# Embed all chunks with BGE-M3
import faiss

print(f"Embedding {len(chunks_df)} chunks with BGE-M3...")
print("(This will take a few minutes — encoding ~24K text passages)")
start = time.time()

# BGE-M3 doesn't strictly require prefixes, but they can help
# We'll encode in batches to manage memory
all_texts = chunks_df['chunk_text'].tolist()

# Encode in batches of 256 to avoid OOM
batch_size = 256
all_embeddings = []

for i in range(0, len(all_texts), batch_size):
    batch = all_texts[i:i + batch_size]
    batch_emb = embed_model.encode(batch, show_progress_bar=False)
    all_embeddings.append(batch_emb)

    if (i // batch_size + 1) % 10 == 0:
        print(f"  Encoded {min(i + batch_size, len(all_texts))}/{len(all_texts)} chunks")

import numpy as np
all_embeddings = np.vstack(all_embeddings).astype('float32')

elapsed = time.time() - start
print(f"\nDone in {elapsed:.1f}s")
print(f"Embeddings shape: {all_embeddings.shape}")
print(f"  = {all_embeddings.shape[0]} chunks × {all_embeddings.shape[1]} dimensions")

Embedding 23663 chunks with BGE-M3...
(This will take a few minutes — encoding ~24K text passages)
  Encoded 2560/23663 chunks
  Encoded 5120/23663 chunks
  Encoded 7680/23663 chunks
  Encoded 10240/23663 chunks
  Encoded 12800/23663 chunks
  Encoded 15360/23663 chunks
  Encoded 17920/23663 chunks
  Encoded 20480/23663 chunks
  Encoded 23040/23663 chunks

Done in 221.1s
Embeddings shape: (23663, 1024)
  = 23663 chunks × 1024 dimensions


In [13]:
# Build a FAISS index for each patient
# FAISS uses cosine similarity

patient_ids = chunks_df['subject_id'].unique()
patient_indices = {}   # {patient_id: faiss_index}
patient_chunks = {}    # {patient_id: dataframe of chunks}
patient_embeddings = {} # {patient_id: numpy array of embeddings}

print(f"Building FAISS indices for {len(patient_ids)} patients...")

for pid in patient_ids:
    # Get this patient's chunks and embeddings
    mask = chunks_df['subject_id'] == pid
    p_chunks = chunks_df[mask].reset_index(drop=True)
    p_emb = all_embeddings[mask.values]

    # Normalize embeddings for cosine similarity
    # (FAISS IndexFlatIP on normalized vectors = cosine similarity)
    faiss.normalize_L2(p_emb)

    # Build the index
    index = faiss.IndexFlatIP(p_emb.shape[1])  # Inner Product on normalized = cosine
    index.add(p_emb)

    patient_indices[pid] = index
    patient_chunks[pid] = p_chunks
    patient_embeddings[pid] = p_emb

print(f"Done! Built {len(patient_indices)} patient indices")

# Show some examples
for pid in list(patient_ids[:5]):
    n_chunks = patient_indices[pid].ntotal
    print(f"  Patient {pid}: {n_chunks} chunks indexed")

Building FAISS indices for 100 patients...
Done! Built 100 patient indices
  Patient 1084: 70 chunks indexed
  Patient 4954: 248 chunks indexed
  Patient 5954: 45 chunks indexed
  Patient 6214: 348 chunks indexed
  Patient 8501: 47 chunks indexed


## 4. Retrieval — Finding Relevant Chunks

For each patient, we query their FAISS index with a BHC-focused query
and retrieve the top-K most relevant chunks.

A patient with 232K tokens of notes doesn't need all of it
to write a good BHC. They need the admission reason, major diagnoses, key procedures,
important lab trends, and discharge status (maybe 8K–12K tokens of focused content).

In [14]:
# The retrieval query
BHC_QUERY = (
    "Hospital course summary: reason for admission, primary diagnoses, "
    "major procedures and interventions, significant clinical events, "
    "complications, treatment response, condition at discharge, "
    "discharge plan and follow-up"
)

def retrieve_chunks(patient_id, query, embed_model, patient_indices,
                    patient_chunks, top_k=40):
    """
    Retrieve the most relevant chunks for a patient using semantic search.

    Args:
        patient_id: The patient to search
        query: What we're looking for (our BHC-focused query)
        embed_model: BGE-M3 model to encode the query
        patient_indices: Dict of FAISS indices
        patient_chunks: Dict of chunk dataframes
        top_k: How many chunks to retrieve

    Returns:
        retrieved_df: DataFrame of top-K chunks with similarity scores
        total_tokens: Total tokens in retrieved chunks
    """
    # Encode the query
    query_emb = embed_model.encode([query]).astype('float32')
    faiss.normalize_L2(query_emb)

    # Search this patient's index
    scores, indices = patient_indices[patient_id].search(query_emb, top_k)

    # Get the matching chunks
    p_chunks = patient_chunks[patient_id]
    retrieved = p_chunks.iloc[indices[0]].copy()
    retrieved['similarity_score'] = scores[0]

    # Sort chronologically — the LLM should see events in order
    retrieved = retrieved.sort_values('chartdate')

    total_tokens = retrieved['chunk_tokens'].sum()

    return retrieved, total_tokens

In [15]:
# Test on patient 1084 (same one we tested in baseline)
test_pid = human_bhcs['subject_id'].iloc[0]

retrieved, total_tokens = retrieve_chunks(
    test_pid, BHC_QUERY, embed_model, patient_indices, patient_chunks, top_k=40
)

print(f"Patient {test_pid}: Retrieved {len(retrieved)} chunks ({total_tokens} tokens)")
print(f"Original notes: {notes[notes['SUBJECT_ID'] == test_pid]['TEXT'].apply(lambda x: len(tokenizer.encode(x))).sum()} tokens")
print(f"Compression ratio: {total_tokens / notes[notes['SUBJECT_ID'] == test_pid]['TEXT'].apply(lambda x: len(tokenizer.encode(x))).sum():.1%}")

print(f"\nTop 5 chunks by similarity score:")
top5 = retrieved.sort_values('similarity_score', ascending=False).head()
for _, chunk in top5.iterrows():
    print(f"\n  Score: {chunk['similarity_score']:.3f} | {chunk['category']} — {chunk['chartdate']}")
    print(f"  {chunk['chunk_text'][:150]}...")

Patient 1084: Retrieved 40 chunks (11102 tokens)
Original notes: 15988 tokens
Compression ratio: 69.4%

Top 5 chunks by similarity score:

  Score: 0.614 | Physician — 2198-08-03
  Propofol - 40 mcg/Kg/min
   Other ICU medications:
   Lorazepam (Ativan) - [**2198-8-2**] 08:45 PM
   Fentanyl - [**2198-8-3**] 12:00 AM
   Other medi...

  Score: 0.614 | Physician — 2198-08-03
  Propofol - 40 mcg/Kg/min
   Other ICU medications:
   Lorazepam (Ativan) - [**2198-8-2**] 08:45 PM
   Fentanyl - [**2198-8-3**] 12:00 AM
   Other medi...

  Score: 0.571 | Physician — 2198-08-03
  ubated this am.
   History obtained from Medical records
   Allergies:
   No Known Drug Allergies
   Last dose of Antibiotics:
   Vancomycin - [**2198...

  Score: 0.563 | Physician — 2198-08-03
    ICU Care
   Nutrition: start po when more awake
   Glycemic Control:
   Lines:
   18 Gauge - [**2198-8-2**] 07:05 PM
   Prophylaxis:
   DVT: SQ UF H...

  Score: 0.563 | Physician — 2198-08-03
    ICU Care
   Nutrition: start 

In [16]:
def retrieve_chunks_dedup(patient_id, query, embed_model, patient_indices,
                          patient_chunks, top_k=40, fetch_k=80):
    """
    Retrieve top-K chunks with deduplication.

    We fetch more than we need (fetch_k), then deduplicate by removing
    chunks that overlap heavily with already-selected chunks.

    Args:
        top_k: How many unique chunks we want
        fetch_k: How many to initially fetch before deduplication
    """
    # Encode the query
    query_emb = embed_model.encode([query]).astype('float32')
    faiss.normalize_L2(query_emb)

    # Fetch more than we need
    p_chunks = patient_chunks[patient_id]
    actual_fetch = min(fetch_k, patient_indices[patient_id].ntotal)
    scores, indices = patient_indices[patient_id].search(query_emb, actual_fetch)

    # Deduplicate: skip chunks that overlap heavily with already-selected ones
    selected = []
    selected_texts = set()

    for idx, score in zip(indices[0], scores[0]):
        chunk = p_chunks.iloc[idx]

        # Create a fingerprint from the middle portion of the chunk
        # (the start/end may overlap with neighbors, but the middle is unique)
        text = chunk['chunk_text']
        mid = len(text) // 2
        fingerprint = text[max(0, mid-100):mid+100]

        # Check if we've already selected a chunk with very similar content
        is_duplicate = False
        for seen in selected_texts:
            # If >60% of the fingerprint appears in an already-selected chunk
            overlap = len(set(fingerprint.split()) & set(seen.split()))
            total = max(len(set(fingerprint.split())), 1)
            if overlap / total > 0.6:
                is_duplicate = True
                break

        if not is_duplicate:
            selected.append({**chunk.to_dict(), 'similarity_score': score})
            selected_texts.add(fingerprint)

        if len(selected) >= top_k:
            break

    retrieved = pd.DataFrame(selected)

    # Sort chronologically for the LLM
    retrieved = retrieved.sort_values('chartdate')
    total_tokens = retrieved['chunk_tokens'].sum()

    return retrieved, total_tokens


# Test the improved version on the same patient
retrieved, total_tokens = retrieve_chunks_dedup(
    test_pid, BHC_QUERY, embed_model, patient_indices, patient_chunks, top_k=40
)

print(f"Patient {test_pid}: Retrieved {len(retrieved)} unique chunks ({total_tokens} tokens)")

print(f"\nTop 5 chunks by similarity (after dedup):")
top5 = retrieved.sort_values('similarity_score', ascending=False).head()
for _, chunk in top5.iterrows():
    print(f"\n  Score: {chunk['similarity_score']:.3f} | {chunk['category']} — {chunk['chartdate']}")
    print(f"  {chunk['chunk_text'][:150]}...")

# What note categories got retrieved?
print(f"\nRetrieved note categories:")
cat_counts = retrieved['category'].value_counts()
for cat, count in cat_counts.items():
    print(f"  {cat}: {count} chunks")

Patient 1084: Retrieved 40 unique chunks (10840 tokens)

Top 5 chunks by similarity (after dedup):

  Score: 0.614 | Physician — 2198-08-03
  Propofol - 40 mcg/Kg/min
   Other ICU medications:
   Lorazepam (Ativan) - [**2198-8-2**] 08:45 PM
   Fentanyl - [**2198-8-3**] 12:00 AM
   Other medi...

  Score: 0.571 | Physician — 2198-08-03
  ubated this am.
   History obtained from Medical records
   Allergies:
   No Known Drug Allergies
   Last dose of Antibiotics:
   Vancomycin - [**2198...

  Score: 0.563 | Physician — 2198-08-03
    ICU Care
   Nutrition: start po when more awake
   Glycemic Control:
   Lines:
   18 Gauge - [**2198-8-2**] 07:05 PM
   Prophylaxis:
   DVT: SQ UF H...

  Score: 0.562 | Physician — 2198-08-02
  
   Prophylaxis: Subutaneous heparin, pneumoboots, bowel reg, PPI
   .
   Access: peripherals, 2 18s
   .
   Code:  presumed full
   .
   Communicatio...

  Score: 0.549 | Physician — 2198-08-03
  598**]-will check TSH as may be playing a role in pt's AMS.
   -TSH
  

## 5. RAG Generation

Feed retrieved chunks to Mistral 7B with a focused prompt.
Same model, same generation settings as baseline. The ONLY difference
is that the input is retrieved relevant content instead of everything concatenated.

In [17]:
# We tell the model what it's receiving (retrieved relevant excerpts, not full notes)
RAG_PROMPT = """You are a physician. Based on the following relevant excerpts from a patient's clinical notes during their hospital stay, write a Brief Hospital Course summarizing the key events, findings, treatments, and outcomes.

These excerpts have been selected as the most relevant portions from the patient's full medical record. They are presented in chronological order.

Relevant Clinical Excerpts:
{context}

Brief Hospital Course:"""


def build_rag_context(retrieved_df):
    """
    Format retrieved chunks into a context string for the LLM.
    Each chunk gets a header showing its source note type and date.
    """
    sections = []
    for _, chunk in retrieved_df.iterrows():
        header = f"[{chunk['category']} — {chunk['chartdate']}]"
        sections.append(f"{header}\n{chunk['chunk_text']}")

    return "\n\n".join(sections)


def generate_bhc_rag(retrieved_df, model, tokenizer, max_new_tokens=1024):
    """
    Generate a BHC from retrieved chunks.
    Uses the same generation settings as baseline for fair comparison.

    Returns:
        generated_text: The model's BHC output
        input_tokens: Number of tokens in the prompt
    """
    # Build context from retrieved chunks
    context = build_rag_context(retrieved_df)
    full_prompt = RAG_PROMPT.format(context=context)

    # Format as chat message (same as baseline)
    messages = [{"role": "user", "content": full_prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Tokenize
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    input_tokens = inputs['input_ids'].shape[1]

    # Generate with IDENTICAL settings to baseline
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1
        )

    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return generated_text, input_tokens

In [18]:
# Test on our example patient first
print(f"Generating RAG-based BHC for patient {test_pid}...")
print(f"Retrieved context: {total_tokens} tokens (vs {notes[notes['SUBJECT_ID'] == test_pid]['TEXT'].apply(lambda x: len(tokenizer.encode(x))).sum()} total)")

generated_rag, input_tokens = generate_bhc_rag(retrieved, model, tokenizer)

print(f"Input tokens (prompt + context): {input_tokens}")
print(f"Generated BHC ({len(tokenizer.encode(generated_rag))} tokens):")
print("=" * 80)
print(generated_rag)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Generating RAG-based BHC for patient 1084...
Retrieved context: 10840 tokens (vs 15988 total)
Input tokens (prompt + context): 11705
Generated BHC (454 tokens):
The patient, a 61-year-old male with a history of metastatic prostate cancer, pelvic osteoarthritis, recurrent urinary tract infections (UTIs), and chronic pain, presented to the emergency department (ED) in an altered mental state. The patient was found by construction workers at home acting strangely and was brought to the ED. In the ED, the patient was intubated due to agitation and given Narcan, which escalated the patient's altered mental status.

The patient was transferred to the Medical Intensive Care Unit (MICU) intubated, sedated, and hemodynamically stable. Initial lab results were unremarkable, and radiology studies, including a head CT, chest X-ray, and EKG, were also unremarkable. The patient's ileal conduit was noted to contain clear urine, and the lungs were clear.

The patient was diagnosed with respiratory fai

In [19]:
# Run RAG pipeline on all 100 patients
rag_results = []

print(f"Running RAG pipeline on {len(human_bhcs)} patients...")
print("-" * 60)

start_total = time.time()

for i, row in human_bhcs.iterrows():
    patient_id = row['subject_id']

    # Step 1: Retrieve relevant chunks
    retrieved_df, retrieved_tokens = retrieve_chunks_dedup(
        patient_id, BHC_QUERY, embed_model, patient_indices, patient_chunks, top_k=40
    )

    # Step 2: Generate BHC from retrieved chunks
    start = time.time()
    generated, input_tokens = generate_bhc_rag(retrieved_df, model, tokenizer)
    elapsed = time.time() - start

    # Get total tokens in the patient's full notes (for comparison)
    full_tokens = notes[notes['SUBJECT_ID'] == patient_id]['TEXT'].apply(
        lambda x: len(tokenizer.encode(x))
    ).sum()

    # Was this patient truncated in baseline?
    was_truncated_baseline = baseline_results[
        baseline_results['subject_id'] == patient_id
    ]['was_truncated'].iloc[0]

    rag_results.append({
        'subject_id': patient_id,
        'full_note_tokens': full_tokens,
        'retrieved_tokens': retrieved_tokens,
        'input_tokens': input_tokens,
        'compression_ratio': retrieved_tokens / full_tokens,
        'was_truncated_baseline': was_truncated_baseline,
        'generated_bhc': generated,
        'generated_tokens': len(tokenizer.encode(generated)),
        'generation_time_sec': round(elapsed, 1),
        'human_bhc': row['brief_hospital_course'],
        'n_chunks_retrieved': len(retrieved_df)
    })

    # Progress update every 10 patients
    if len(rag_results) % 10 == 0:
        print(f"  {len(rag_results)}/100 | Last: {full_tokens} full → "
              f"{retrieved_tokens} retrieved ({retrieved_tokens/full_tokens:.0%}) | "
              f"{elapsed:.1f}s | Baseline truncated: {was_truncated_baseline}")

total_time = time.time() - start_total
print("-" * 60)
print(f"Done! Total time: {total_time/60:.1f} minutes")

rag_results_df = pd.DataFrame(rag_results)

# Save
rag_results_df.to_parquet(f"{DATA_DIR}/rag_results.parquet", index=False)
print(f"Saved {len(rag_results_df)} results to Drive")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Running RAG pipeline on 100 patients...
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  10/100 | Last: 95397 full → 10857 retrieved (11%) | 84.9s | Baseline truncated: True


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

  80/100 | Last: 23144 full → 10392 retrieved (45%) | 43.3s | Baseline truncated: False


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  90/100 | Last: 22100 full → 10994 retrieved (50%) | 30.6s | Baseline truncated: False


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  100/100 | Last: 26950 full → 10540 retrieved (39%) | 24.1s | Baseline truncated: False
------------------------------------------------------------
Done! Total time: 62.2 minutes
Saved 100 results to Drive


## 6. Evaluation — RAG vs Baseline

Same metrics as baseline (ROUGE-1, ROUGE-2, ROUGE-L, BERTScore) for a
direct apples-to-apples comparison. Then subgroup analysis to see if RAG
helps most where baseline struggled (the 42 truncated patients).

In [21]:
!pip install -q rouge-score bert-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.8 MB/s eta 0:00:00


In [22]:
from rouge_score import rouge_scorer

# Initialize ROUGE scorer (same as baseline)
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Compute ROUGE scores for all RAG-generated BHCs
rag_rouge = []
for _, row in rag_results_df.iterrows():
    scores = scorer.score(row['human_bhc'], row['generated_bhc'])
    rag_rouge.append({
        'subject_id': row['subject_id'],
        'rouge1_f': scores['rouge1'].fmeasure,
        'rouge2_f': scores['rouge2'].fmeasure,
        'rougeL_f': scores['rougeL'].fmeasure,
    })

rag_rouge_df = pd.DataFrame(rag_rouge)

print("RAG ROUGE Scores (all 100 patients):")
print(f"  ROUGE-1 F1: {rag_rouge_df['rouge1_f'].mean():.3f} (±{rag_rouge_df['rouge1_f'].std():.3f})")
print(f"  ROUGE-2 F1: {rag_rouge_df['rouge2_f'].mean():.3f} (±{rag_rouge_df['rouge2_f'].std():.3f})")
print(f"  ROUGE-L F1: {rag_rouge_df['rougeL_f'].mean():.3f} (±{rag_rouge_df['rougeL_f'].std():.3f})")

RAG ROUGE Scores (all 100 patients):
  ROUGE-1 F1: 0.339 (±0.072)
  ROUGE-2 F1: 0.079 (±0.035)
  ROUGE-L F1: 0.163 (±0.039)


In [23]:
from bert_score import score as bert_score_fn

print("Computing BERTScore (this may take a few minutes)...")

P, R, F1 = bert_score_fn(
    rag_results_df['generated_bhc'].tolist(),
    rag_results_df['human_bhc'].tolist(),
    lang='en',
    verbose=True
)

rag_results_df['bertscore_p'] = P.numpy()
rag_results_df['bertscore_r'] = R.numpy()
rag_results_df['bertscore_f1'] = F1.numpy()

print(f"\nRAG BERTScore (all 100 patients):")
print(f"  Precision: {rag_results_df['bertscore_p'].mean():.3f} (±{rag_results_df['bertscore_p'].std():.3f})")
print(f"  Recall:    {rag_results_df['bertscore_r'].mean():.3f} (±{rag_results_df['bertscore_r'].std():.3f})")
print(f"  F1:        {rag_results_df['bertscore_f1'].mean():.3f} (±{rag_results_df['bertscore_f1'].std():.3f})")

Computing BERTScore (this may take a few minutes)...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 3.21 seconds, 31.11 sentences/sec

RAG BERTScore (all 100 patients):
  Precision: 0.830 (±0.014)
  Recall:    0.807 (±0.016)
  F1:        0.818 (±0.012)


In [24]:
# Merge RAG ROUGE scores into results
rag_eval = rag_results_df.merge(rag_rouge_df, on='subject_id')

# Save evaluation results
rag_eval.to_parquet(f"{DATA_DIR}/rag_eval_results.parquet", index=False)
print("Saved RAG evaluation results to Drive\n")

# HEAD-TO-HEAD: Baseline vs RAG (full cohort)
print("=" * 65)
print("BASELINE vs RAG — Full Cohort (n=100)")
print("=" * 65)
print(f"{'Metric':<20} {'Baseline':<22} {'RAG':<22} {'Δ'}")
print("-" * 65)

for metric, bl_col, rag_col in [
    ('ROUGE-1 F1', 'rouge1_f', 'rouge1_f'),
    ('ROUGE-2 F1', 'rouge2_f', 'rouge2_f'),
    ('ROUGE-L F1', 'rougeL_f', 'rougeL_f'),
]:
    bl_val = baseline_eval[bl_col].mean()
    rag_val = rag_eval[rag_col].mean()
    delta = rag_val - bl_val
    sign = "+" if delta >= 0 else ""
    print(f"{metric:<20} {bl_val:.3f}                {rag_val:.3f}                {sign}{delta:.3f}")

bl_bert = baseline_eval['bertscore_f1'].mean()
rag_bert = rag_eval['bertscore_f1'].mean()
delta = rag_bert - bl_bert
sign = "+" if delta >= 0 else ""
print(f"{'BERTScore F1':<20} {bl_bert:.3f}                {rag_bert:.3f}                {sign}{delta:.3f}")

Saved RAG evaluation results to Drive

BASELINE vs RAG — Full Cohort (n=100)
Metric               Baseline               RAG                    Δ
-----------------------------------------------------------------
ROUGE-1 F1           0.339                0.339                -0.001
ROUGE-2 F1           0.084                0.079                -0.005
ROUGE-L F1           0.165                0.163                -0.002
BERTScore F1         0.819                0.818                -0.000


In [25]:
#Split by baseline truncation status

# Merge baseline and RAG scores per patient
comparison = baseline_eval[['subject_id', 'rouge1_f', 'rouge2_f', 'rougeL_f',
                            'bertscore_f1', 'was_truncated_x']].rename(
    columns={'rouge1_f': 'bl_rouge1', 'rouge2_f': 'bl_rouge2',
             'rougeL_f': 'bl_rougeL', 'bertscore_f1': 'bl_bertscore',
             'was_truncated_x': 'was_truncated_baseline'}
)

comparison = comparison.merge(
    rag_eval[['subject_id', 'rouge1_f', 'rouge2_f', 'rougeL_f', 'bertscore_f1']].rename(
        columns={'rouge1_f': 'rag_rouge1', 'rouge2_f': 'rag_rouge2',
                 'rougeL_f': 'rag_rougeL', 'bertscore_f1': 'rag_bertscore'}
    ),
    on='subject_id'
)

# Compute per-patient deltas
comparison['delta_rouge1'] = comparison['rag_rouge1'] - comparison['bl_rouge1']
comparison['delta_rougeL'] = comparison['rag_rougeL'] - comparison['bl_rougeL']
comparison['delta_bertscore'] = comparison['rag_bertscore'] - comparison['bl_bertscore']

# Split by truncation status
trunc = comparison[comparison['was_truncated_baseline'] == True]
no_trunc = comparison[comparison['was_truncated_baseline'] == False]

print("=" * 70)
print("SUBGROUP ANALYSIS: Did RAG help truncated patients?")
print("=" * 70)

print(f"\n{'PREVIOUSLY TRUNCATED (n=42)':}")
print(f"  These patients lost data in baseline due to 32K context limit")
print(f"  {'Metric':<16} {'Baseline':<14} {'RAG':<14} {'Δ':<10} {'Improved?'}")
print(f"  {'-'*58}")
print(f"  {'ROUGE-1 F1':<16} {trunc['bl_rouge1'].mean():.3f}          "
      f"{trunc['rag_rouge1'].mean():.3f}          "
      f"{trunc['delta_rouge1'].mean():+.3f}      "
      f"{'✓' if trunc['delta_rouge1'].mean() > 0 else '✗'}")
print(f"  {'ROUGE-L F1':<16} {trunc['bl_rougeL'].mean():.3f}          "
      f"{trunc['rag_rougeL'].mean():.3f}          "
      f"{trunc['delta_rougeL'].mean():+.3f}      "
      f"{'✓' if trunc['delta_rougeL'].mean() > 0 else '✗'}")
print(f"  {'BERTScore F1':<16} {trunc['bl_bertscore'].mean():.3f}          "
      f"{trunc['rag_bertscore'].mean():.3f}          "
      f"{trunc['delta_bertscore'].mean():+.3f}      "
      f"{'✓' if trunc['delta_bertscore'].mean() > 0 else '✗'}")

print(f"\n{'NOT TRUNCATED (n=58)':}")
print(f"  These patients already had full data in baseline")
print(f"  {'Metric':<16} {'Baseline':<14} {'RAG':<14} {'Δ':<10} {'Improved?'}")
print(f"  {'-'*58}")
print(f"  {'ROUGE-1 F1':<16} {no_trunc['bl_rouge1'].mean():.3f}          "
      f"{no_trunc['rag_rouge1'].mean():.3f}          "
      f"{no_trunc['delta_rouge1'].mean():+.3f}      "
      f"{'✓' if no_trunc['delta_rouge1'].mean() > 0 else '✗'}")
print(f"  {'ROUGE-L F1':<16} {no_trunc['bl_rougeL'].mean():.3f}          "
      f"{no_trunc['rag_rougeL'].mean():.3f}          "
      f"{no_trunc['delta_rougeL'].mean():+.3f}      "
      f"{'✓' if no_trunc['delta_rougeL'].mean() > 0 else '✗'}")
print(f"  {'BERTScore F1':<16} {no_trunc['bl_bertscore'].mean():.3f}          "
      f"{no_trunc['rag_bertscore'].mean():.3f}          "
      f"{no_trunc['delta_bertscore'].mean():+.3f}      "
      f"{'✓' if no_trunc['delta_bertscore'].mean() > 0 else '✗'}")

# How many patients improved vs got worse?
print(f"\n{'PER-PATIENT WINS/LOSSES (ROUGE-L)':}")
print(f"  Truncated:     {(trunc['delta_rougeL'] > 0).sum()}/42 improved, "
      f"{(trunc['delta_rougeL'] < 0).sum()}/42 worsened, "
      f"{(trunc['delta_rougeL'] == 0).sum()}/42 tied")
print(f"  Not truncated: {(no_trunc['delta_rougeL'] > 0).sum()}/58 improved, "
      f"{(no_trunc['delta_rougeL'] < 0).sum()}/58 worsened, "
      f"{(no_trunc['delta_rougeL'] == 0).sum()}/58 tied")

SUBGROUP ANALYSIS: Did RAG help truncated patients?

PREVIOUSLY TRUNCATED (n=42)
  These patients lost data in baseline due to 32K context limit
  Metric           Baseline       RAG            Δ          Improved?
  ----------------------------------------------------------
  ROUGE-1 F1       0.314          0.310          -0.004      ✗
  ROUGE-L F1       0.148          0.143          -0.005      ✗
  BERTScore F1     0.814          0.813          -0.001      ✗

NOT TRUNCATED (n=58)
  These patients already had full data in baseline
  Metric           Baseline       RAG            Δ          Improved?
  ----------------------------------------------------------
  ROUGE-1 F1       0.358          0.359          +0.001      ✓
  ROUGE-L F1       0.177          0.176          -0.000      ✗
  BERTScore F1     0.823          0.823          +0.000      ✓

PER-PATIENT WINS/LOSSES (ROUGE-L)
  Truncated:     16/42 improved, 26/42 worsened, 0/42 tied
  Not truncated: 31/58 improved, 27/58 worsened,

## 6.1 Improved RAG: Multi-Query Retrieval

The single generic query retrieved too much templated content (med lists,
ICU order sets) and missed narrative content.

Let's try to use multiple targeted queries, one for each component of a BHC,
mirroring how a physician mentally organizes a hospital course.
This is clinically motivated: you don't think "hospital course" as one blob,
you think "why did they come in? what did we find? what did we do? how did they do?"

In [26]:
# Instead of one generic query, use multiple specific queries that mirror
# how a physician actually thinks about a BHC
# Each query targets a different aspect of the hospital course

BHC_QUERIES = {
    'admission': "Reason for admission, chief complaint, presenting symptoms, emergency department evaluation",
    'diagnoses': "Primary diagnosis, secondary diagnoses, differential diagnosis, assessment",
    'workup': "Laboratory results, imaging findings, diagnostic test results, pathology",
    'treatment': "Medications started, surgical procedures, interventions performed, treatments administered",
    'course': "Clinical progression, complications, response to treatment, significant events during hospitalization",
    'discharge': "Condition at discharge, discharge disposition, follow-up plan, discharge medications"
}

def retrieve_multi_query(patient_id, queries, embed_model, patient_indices,
                         patient_chunks, chunks_per_query=10, max_total_tokens=12000):
    """
    Retrieve chunks using multiple targeted queries, then deduplicate.

    This mirrors how a physician thinks about a BHC — not as one blob,
    but as admission → workup → treatment → course → discharge.

    Args:
        queries: Dict of {section_name: query_text}
        chunks_per_query: How many chunks to retrieve per query
        max_total_tokens: Token budget for total retrieved context
    """
    all_selected = []
    seen_fingerprints = set()

    for section, query in queries.items():
        # Encode the query
        query_emb = embed_model.encode([query]).astype('float32')
        faiss.normalize_L2(query_emb)

        # Search
        p_chunks = patient_chunks[patient_id]
        actual_fetch = min(chunks_per_query * 2, patient_indices[patient_id].ntotal)
        scores, indices = patient_indices[patient_id].search(query_emb, actual_fetch)

        # Deduplicate against all previously selected chunks
        section_count = 0
        for idx, score in zip(indices[0], scores[0]):
            if section_count >= chunks_per_query:
                break

            chunk = p_chunks.iloc[idx]
            text = chunk['chunk_text']
            mid = len(text) // 2
            fingerprint = text[max(0, mid-100):mid+100]

            is_dup = False
            for seen in seen_fingerprints:
                overlap = len(set(fingerprint.split()) & set(seen.split()))
                total = max(len(set(fingerprint.split())), 1)
                if overlap / total > 0.6:
                    is_dup = True
                    break

            if not is_dup:
                all_selected.append({
                    **chunk.to_dict(),
                    'similarity_score': score,
                    'query_section': section
                })
                seen_fingerprints.add(fingerprint)
                section_count += 1

    retrieved = pd.DataFrame(all_selected)

    # Enforce token budget — keep highest-scoring chunks if over budget
    retrieved = retrieved.sort_values('similarity_score', ascending=False)
    cumulative_tokens = retrieved['chunk_tokens'].cumsum()
    retrieved = retrieved[cumulative_tokens <= max_total_tokens]

    # Re-sort chronologically for the LLM
    retrieved = retrieved.sort_values('chartdate')
    total_tokens = retrieved['chunk_tokens'].sum()

    return retrieved, total_tokens


# Test on our example patient
retrieved_mq, total_tokens_mq = retrieve_multi_query(
    test_pid, BHC_QUERIES, embed_model, patient_indices, patient_chunks
)

print(f"Patient {test_pid}: {len(retrieved_mq)} chunks, {total_tokens_mq} tokens")
print(f"\nChunks per query section:")
print(retrieved_mq['query_section'].value_counts().to_string())
print(f"\nNote categories retrieved:")
print(retrieved_mq['category'].value_counts().to_string())

Patient 1084: 31 chunks, 8480 tokens

Chunks per query section:
query_section
admission    10
diagnoses     9
workup        6
treatment     3
course        2
discharge     1

Note categories retrieved:
category
Physician    25
Nursing       6


In [27]:
# Run improved multi-query RAG on all 100 patients
rag2_results = []

print(f"Running multi-query RAG on {len(human_bhcs)} patients...")
print("-" * 60)

start_total = time.time()

for i, row in human_bhcs.iterrows():
    patient_id = row['subject_id']

    # Multi-query retrieval
    retrieved_df, retrieved_tokens = retrieve_multi_query(
        patient_id, BHC_QUERIES, embed_model, patient_indices, patient_chunks
    )

    # Generate BHC (same function, same settings)
    start = time.time()
    generated, input_tokens = generate_bhc_rag(retrieved_df, model, tokenizer)
    elapsed = time.time() - start

    full_tokens = notes[notes['SUBJECT_ID'] == patient_id]['TEXT'].apply(
        lambda x: len(tokenizer.encode(x))
    ).sum()

    was_truncated_baseline = baseline_results[
        baseline_results['subject_id'] == patient_id
    ]['was_truncated'].iloc[0]

    rag2_results.append({
        'subject_id': patient_id,
        'full_note_tokens': full_tokens,
        'retrieved_tokens': retrieved_tokens,
        'input_tokens': input_tokens,
        'compression_ratio': retrieved_tokens / full_tokens,
        'was_truncated_baseline': was_truncated_baseline,
        'generated_bhc': generated,
        'generated_tokens': len(tokenizer.encode(generated)),
        'generation_time_sec': round(elapsed, 1),
        'human_bhc': row['brief_hospital_course'],
        'n_chunks_retrieved': len(retrieved_df)
    })

    if len(rag2_results) % 10 == 0:
        print(f"  {len(rag2_results)}/100 | Last: {full_tokens} full → "
              f"{retrieved_tokens} retrieved ({retrieved_tokens/full_tokens:.0%}) | "
              f"{elapsed:.1f}s | Baseline truncated: {was_truncated_baseline}")

total_time = time.time() - start_total
print("-" * 60)
print(f"Done! Total time: {total_time/60:.1f} minutes")

rag2_results_df = pd.DataFrame(rag2_results)
rag2_results_df.to_parquet(f"{DATA_DIR}/rag2_multi_query_results.parquet", index=False)
print(f"Saved {len(rag2_results_df)} results to Drive")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Running multi-query RAG on 100 patients...
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  10/100 | Last: 95397 full → 11822 retrieved (12%) | 30.7s | Baseline truncated: True


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

  80/100 | Last: 23144 full → 9745 retrieved (42%) | 49.0s | Baseline truncated: False


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  90/100 | Last: 22100 full → 11234 retrieved (51%) | 27.3s | Baseline truncated: False


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  100/100 | Last: 26950 full → 8339 retrieved (31%) | 80.4s | Baseline truncated: False
------------------------------------------------------------
Done! Total time: 65.0 minutes
Saved 100 results to Drive


In [28]:
# ROUGE scores for multi-query RAG
rag2_rouge = []
for _, row in rag2_results_df.iterrows():
    scores = scorer.score(row['human_bhc'], row['generated_bhc'])
    rag2_rouge.append({
        'subject_id': row['subject_id'],
        'rouge1_f': scores['rouge1'].fmeasure,
        'rouge2_f': scores['rouge2'].fmeasure,
        'rougeL_f': scores['rougeL'].fmeasure,
    })

rag2_rouge_df = pd.DataFrame(rag2_rouge)

# BERTScore
print("Computing BERTScore for multi-query RAG...")
P, R, F1 = bert_score_fn(
    rag2_results_df['generated_bhc'].tolist(),
    rag2_results_df['human_bhc'].tolist(),
    lang='en',
    verbose=True
)

rag2_results_df['bertscore_p'] = P.numpy()
rag2_results_df['bertscore_r'] = R.numpy()
rag2_results_df['bertscore_f1'] = F1.numpy()

# Merge and save
rag2_eval = rag2_results_df.merge(rag2_rouge_df, on='subject_id')
rag2_eval.to_parquet(f"{DATA_DIR}/rag2_multi_query_eval_results.parquet", index=False)
print("Saved multi-query RAG evaluation results to Drive")

Computing BERTScore for multi-query RAG...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 3.15 seconds, 31.77 sentences/sec
Saved multi-query RAG evaluation results to Drive


In [29]:
# THREE-WAY COMPARISON: Baseline vs RAG v1 vs RAG v2

print("=" * 75)
print("FULL COMPARISON — All Three Approaches (n=100)")
print("=" * 75)
print(f"{'Metric':<16} {'Baseline':<18} {'RAG single-Q':<18} {'RAG multi-Q':<18}")
print("-" * 75)

metrics = [
    ('ROUGE-1 F1', baseline_eval['rouge1_f'], rag_eval['rouge1_f'], rag2_eval['rouge1_f']),
    ('ROUGE-2 F1', baseline_eval['rouge2_f'], rag_eval['rouge2_f'], rag2_eval['rouge2_f']),
    ('ROUGE-L F1', baseline_eval['rougeL_f'], rag_eval['rougeL_f'], rag2_eval['rougeL_f']),
    ('BERTScore F1', baseline_eval['bertscore_f1'], rag_eval['bertscore_f1'], rag2_eval['bertscore_f1']),
]

for name, bl, r1, r2 in metrics:
    print(f"{name:<16} {bl.mean():.3f} (±{bl.std():.3f})   "
          f"{r1.mean():.3f} (±{r1.std():.3f})   "
          f"{r2.mean():.3f} (±{r2.std():.3f})")

# Generated length comparison
print(f"\n{'Gen. length':<16} {baseline_results['generated_tokens'].mean():.0f} tokens       "
      f"{rag_results_df['generated_tokens'].mean():.0f} tokens       "
      f"{rag2_results_df['generated_tokens'].mean():.0f} tokens")

FULL COMPARISON — All Three Approaches (n=100)
Metric           Baseline           RAG single-Q       RAG multi-Q       
---------------------------------------------------------------------------
ROUGE-1 F1       0.339 (±0.075)   0.339 (±0.072)   0.335 (±0.070)
ROUGE-2 F1       0.084 (±0.040)   0.079 (±0.035)   0.081 (±0.033)
ROUGE-L F1       0.165 (±0.042)   0.163 (±0.039)   0.162 (±0.039)
BERTScore F1     0.819 (±0.014)   0.818 (±0.012)   0.818 (±0.012)

Gen. length      441 tokens       440 tokens       455 tokens


In [30]:
# SUBGROUP: Baseline vs Multi-Query RAG, split by truncation
comparison2 = baseline_eval[['subject_id', 'rouge1_f', 'rouge2_f', 'rougeL_f',
                              'bertscore_f1', 'was_truncated_x']].rename(
    columns={'rouge1_f': 'bl_rouge1', 'rouge2_f': 'bl_rouge2',
             'rougeL_f': 'bl_rougeL', 'bertscore_f1': 'bl_bertscore',
             'was_truncated_x': 'was_truncated_baseline'}
)

comparison2 = comparison2.merge(
    rag2_eval[['subject_id', 'rouge1_f', 'rouge2_f', 'rougeL_f', 'bertscore_f1']].rename(
        columns={'rouge1_f': 'rag2_rouge1', 'rouge2_f': 'rag2_rouge2',
                 'rougeL_f': 'rag2_rougeL', 'bertscore_f1': 'rag2_bertscore'}
    ),
    on='subject_id'
)

comparison2['delta_rouge1'] = comparison2['rag2_rouge1'] - comparison2['bl_rouge1']
comparison2['delta_rougeL'] = comparison2['rag2_rougeL'] - comparison2['bl_rougeL']
comparison2['delta_bertscore'] = comparison2['rag2_bertscore'] - comparison2['bl_bertscore']

trunc2 = comparison2[comparison2['was_truncated_baseline'] == True]
no_trunc2 = comparison2[comparison2['was_truncated_baseline'] == False]

print("=" * 70)
print("SUBGROUP: Baseline vs Multi-Query RAG")
print("=" * 70)

for label, subset in [("PREVIOUSLY TRUNCATED (n=42)", trunc2),
                       ("NOT TRUNCATED (n=58)", no_trunc2)]:
    print(f"\n{label}")
    print(f"  {'Metric':<16} {'Baseline':<14} {'RAG v2':<14} {'Δ':<10} {'Result'}")
    print(f"  {'-'*60}")
    for metric, bl_col, rag_col, d_col in [
        ('ROUGE-1 F1', 'bl_rouge1', 'rag2_rouge1', 'delta_rouge1'),
        ('ROUGE-L F1', 'bl_rougeL', 'rag2_rougeL', 'delta_rougeL'),
        ('BERTScore F1', 'bl_bertscore', 'rag2_bertscore', 'delta_bertscore'),
    ]:
        bl_val = subset[bl_col].mean()
        rag_val = subset[rag_col].mean()
        delta = subset[d_col].mean()
        result = '✓ improved' if delta > 0.001 else ('✗ worse' if delta < -0.001 else '≈ same')
        print(f"  {metric:<16} {bl_val:.3f}          {rag_val:.3f}          {delta:+.3f}      {result}")

print(f"\nPER-PATIENT WINS/LOSSES (ROUGE-L):")
print(f"  Truncated:     {(trunc2['delta_rougeL'] > 0).sum()}/42 improved, "
      f"{(trunc2['delta_rougeL'] < 0).sum()}/42 worsened")
print(f"  Not truncated: {(no_trunc2['delta_rougeL'] > 0).sum()}/58 improved, "
      f"{(no_trunc2['delta_rougeL'] < 0).sum()}/58 worsened")

SUBGROUP: Baseline vs Multi-Query RAG

PREVIOUSLY TRUNCATED (n=42)
  Metric           Baseline       RAG v2         Δ          Result
  ------------------------------------------------------------
  ROUGE-1 F1       0.314          0.315          +0.001      ≈ same
  ROUGE-L F1       0.148          0.147          -0.001      ≈ same
  BERTScore F1     0.814          0.813          -0.001      ✗ worse

NOT TRUNCATED (n=58)
  Metric           Baseline       RAG v2         Δ          Result
  ------------------------------------------------------------
  ROUGE-1 F1       0.358          0.350          -0.008      ✗ worse
  ROUGE-L F1       0.177          0.173          -0.004      ✗ worse
  BERTScore F1     0.823          0.822          -0.000      ≈ same

PER-PATIENT WINS/LOSSES (ROUGE-L):
  Truncated:     20/42 improved, 22/42 worsened
  Not truncated: 31/58 improved, 27/58 worsened


In [31]:
# Qualitative comparison: look at a truncated patient side by side
# Pick a heavily truncated patient to see if RAG changed the content meaningfully

heavy_trunc = comparison2[comparison2['was_truncated_baseline'] == True].sort_values('delta_rougeL')

# Look at one where RAG helped most and one where it hurt most
best_pid = heavy_trunc.iloc[-1]['subject_id']  # Most improved
worst_pid = heavy_trunc.iloc[0]['subject_id']   # Most worsened

for pid, label in [(best_pid, "MOST IMPROVED"), (worst_pid, "MOST WORSENED")]:
    pid = int(pid)
    bl_text = baseline_results[baseline_results['subject_id'] == pid]['generated_bhc'].iloc[0]
    rag_text = rag2_results_df[rag2_results_df['subject_id'] == pid]['generated_bhc'].iloc[0]
    human_text = human_bhcs[human_bhcs['subject_id'] == pid]['brief_hospital_course'].iloc[0]

    full_tokens = notes[notes['SUBJECT_ID'] == pid]['TEXT'].apply(
        lambda x: len(tokenizer.encode(x))
    ).sum()

    bl_rougeL = comparison2[comparison2['subject_id'] == pid]['bl_rougeL'].iloc[0]
    rag_rougeL = comparison2[comparison2['subject_id'] == pid]['rag2_rougeL'].iloc[0]

    print(f"\n{'='*80}")
    print(f"{label} TRUNCATED PATIENT (ID: {pid})")
    print(f"Full notes: {full_tokens} tokens | ROUGE-L: baseline {bl_rougeL:.3f} → RAG {rag_rougeL:.3f}")
    print(f"{'='*80}")
    print(f"\n--- HUMAN BHC ({len(tokenizer.encode(human_text))} tokens) ---")
    print(human_text[:800])
    print(f"\n--- BASELINE ({len(tokenizer.encode(bl_text))} tokens) ---")
    print(bl_text[:800])
    print(f"\n--- RAG v2 ({len(tokenizer.encode(rag_text))} tokens) ---")
    print(rag_text[:800])


MOST IMPROVED TRUNCATED PATIENT (ID: 72940)
Full notes: 37285 tokens | ROUGE-L: baseline 0.113 → RAG 0.240

--- HUMAN BHC (307 tokens) ---
[**10-23**] Mr.[**Known lastname 37742**] was taken to the operating room for an emergent
coronary artery bypass graft x 2 (Left internal mammary artery
grafted to left anterior descending artery/Saphenous vein
grafted to Obtuse Marginal) with Dr.[**Last Name (STitle) **]. Please see
operative report for surgical details. Cross clamp time= 26
minutes. Cardiopulmonary Bypass time= 33 minutes. He was
intubated, sedated, and transferred to the CVICU in critical but
stable condition. Within 24 hours he was weaned from sedation,
awoke neurologically intact and extubated. All lines and drains
were removed in a timely fashion. Beta-blocker/Statin/aspirin ,
and diuresis was initiated. He continued to progress and was
transferred to the telemetry floor for further care. Physical
therapy was consult

--- BASELINE (1025 tokens) ---
**Brief Hospital Course:**


In [32]:
# FINAL RESULTS SUMMARY

print("=" * 75)
print("STEP 3 & 4 COMPLETE — RESULTS SUMMARY")
print("=" * 75)

print("""
MODEL: Mistral 7B Instruct v0.3 (4-bit quantized)
DATASET: VeriFact-BHC (100 MIMIC-III patients, 4,787 notes)
EVALUATION: ROUGE-1/2/L, BERTScore F1 vs human-written BHC

THREE APPROACHES COMPARED:
  1. Baseline: Zero-shot, all notes concatenated (truncated at 32K)
  2. RAG v1: Single generic query, top-40 chunks
  3. RAG v2: Multi-query (6 clinical queries), deduplicated chunks
""")

print("AUTOMATED METRICS (full cohort, n=100):")
print(f"  {'Metric':<16} {'Baseline':<16} {'RAG v1':<16} {'RAG v2':<16}")
print(f"  {'-'*64}")
for name, bl, r1, r2 in metrics:
    print(f"  {name:<16} {bl.mean():.3f}            {r1.mean():.3f}            {r2.mean():.3f}")

print(f"""
KEY FINDINGS:
  1. RAG did not improve automated metrics over baseline on the full cohort.
     All approaches scored within ±0.005 on ROUGE-L and ±0.001 on BERTScore.

  2. Subgroup analysis (truncated vs non-truncated) showed marginal differences.
     RAG v2 improved 20/42 truncated patients but worsened 22/42.

  3. HOWEVER — qualitative review reveals RAG reduced hallucinations:
     - Patient 72940: Baseline hallucinated wrong admission diagnosis;
       RAG correctly identified CABG with specific graft details (LIMA-LAD, SVG-OM)
     - RAG summaries were more focused (avg {rag2_results_df['generated_tokens'].mean():.0f} tokens)
       vs baseline ({baseline_results['generated_tokens'].mean():.0f} tokens)

  4. ROUGE/BERTScore cannot distinguish factual accuracy from plausible-sounding
     but incorrect text — a known limitation in clinical NLP evaluation.

RUNTIME: ~65 min per approach (100 patients on A100)
""")

STEP 3 & 4 COMPLETE — RESULTS SUMMARY

MODEL: Mistral 7B Instruct v0.3 (4-bit quantized)
DATASET: VeriFact-BHC (100 MIMIC-III patients, 4,787 notes)
EVALUATION: ROUGE-1/2/L, BERTScore F1 vs human-written BHC

THREE APPROACHES COMPARED:
  1. Baseline: Zero-shot, all notes concatenated (truncated at 32K)
  2. RAG v1: Single generic query, top-40 chunks
  3. RAG v2: Multi-query (6 clinical queries), deduplicated chunks

AUTOMATED METRICS (full cohort, n=100):
  Metric           Baseline         RAG v1           RAG v2          
  ----------------------------------------------------------------
  ROUGE-1 F1       0.339            0.339            0.335
  ROUGE-2 F1       0.084            0.079            0.081
  ROUGE-L F1       0.165            0.163            0.162
  BERTScore F1     0.819            0.818            0.818

KEY FINDINGS:
  1. RAG did not improve automated metrics over baseline on the full cohort.
     All approaches scored within ±0.005 on ROUGE-L and ±0.001 on BERTScor

## 7. Proposition-Level Evaluation

ROUGE/BERTScore measure word overlap, not factual accuracy.
The VeriFact dataset includes 13,070 clinician-annotated propositions
with ground truth labels (Supported / Not Supported / Not Addressed).

**Approach:** Use the "Supported" propositions from human BHCs as a
fact checklist. For each patient, measure what fraction of verified facts
appear in the baseline vs RAG generated BHCs using semantic similarity.

In [33]:
# Load and explore the proposition annotations
propositions = pd.read_parquet(f"{DATA_DIR}/propositions_with_gt.parquet")

print(f"Total propositions: {len(propositions)}")
print(f"\nColumns: {list(propositions.columns)}")
print(f"\nFirst few rows:")
propositions.head()

Total propositions: 13070

Columns: ['proposition_id', 'text', 'author_type', 'proposition_type', 'parent_text_chunk', 'brief_hospital_course', 'subject_id', 'row_id', 'hadm_id', 'admitdate', 'admittime', 'dischargedate', 'dischargetime', 'human_gt']

First few rows:


,proposition_id,text,author_type,proposition_type,parent_text_chunk,brief_hospital_course,subject_id,row_id,hadm_id,admitdate,admittime,dischargedate,dischargetime,human_gt
0,9be708b0-9e69-722f-4f9d-56951bb20350,The patient has opiate dependence.,human,claim,# Opiate withdrawal - Patient has opiate depen...,Pt is a 61 y.o male with h.o prostate ca with ...,1084,47905.0,194111,2198-08-02,17:41:00,2198-08-04,12:12:00,Supported
1,72f2eecb-03e6-498c-e0e6-b522ab6f5a8e,The patient has been on chronic methadone for ...,human,claim,# Opiate withdrawal - Patient has opiate depen...,Pt is a 61 y.o male with h.o prostate ca with ...,1084,47905.0,194111,2198-08-02,17:41:00,2198-08-04,12:12:00,Supported
2,38ea856e-ede9-7e3b-a77a-cd817ec9f8ac,The patient has been on chronic percocet for p...,human,claim,# Opiate withdrawal - Patient has opiate depen...,Pt is a 61 y.o male with h.o prostate ca with ...,1084,47905.0,194111,2198-08-02,17:41:00,2198-08-04,12:12:00,Supported
3,435d3adc-526c-41a0-f3cd-4b28c71b3eaa,The patient was given narcan by EMS.,human,claim,# Opiate withdrawal - Patient has opiate depen...,Pt is a 61 y.o male with h.o prostate ca with ...,1084,47905.0,194111,2198-08-02,17:41:00,2198-08-04,12:12:00,Supported
4,8a9846be-5f5b-9a1d-bdb0-5bb7c37cf68a,The administration of narcan by EMS escalated ...,human,claim,# Opiate withdrawal - Patient has opiate depen...,Pt is a 61 y.o male with h.o prostate ca with ...,1084,47905.0,194111,2198-08-02,17:41:00,2198-08-04,12:12:00,Supported


In [34]:
# We want: human-authored propositions labeled "Supported"
# These are clinician-verified facts that a good BHC should capture

supported = propositions[
    (propositions['author_type'] == 'human') &
    (propositions['human_gt'] == 'Supported')
].copy()

print(f"Total human propositions: {len(propositions[propositions['author_type'] == 'human'])}")
print(f"Supported (our checklist): {len(supported)}")
print(f"Patients covered: {supported['subject_id'].nunique()}")
print(f"Propositions per patient: mean {supported.groupby('subject_id').size().mean():.1f}, "
      f"median {supported.groupby('subject_id').size().median():.0f}, "
      f"range {supported.groupby('subject_id').size().min()}–{supported.groupby('subject_id').size().max()}")

print(f"\nSample propositions (patient 1084):")
sample = supported[supported['subject_id'] == 1084]['text'].head(10)
for i, text in enumerate(sample, 1):
    print(f"  {i}. {text}")

Total human propositions: 8625
Supported (our checklist): 5396
Patients covered: 100
Propositions per patient: mean 54.0, median 48, range 4–215

Sample propositions (patient 1084):
  1. The patient has opiate dependence.
  2. The patient has been on chronic methadone for pain control.
  3. The patient has been on chronic percocet for pain control.
  4. The patient was given narcan by EMS.
  5. The administration of narcan by EMS escalated the patient's AMS.
  6. The patient's pain medications were initially held while the patient was intubated.
  7. The patient's home regimen of opiates were restarted following extubation.
  8. The patient has a history of prostate cancer.
  9. The patient had surgical intervention for prostate cancer.
  10. The patient has a history of fistulas.


In [35]:
import re

def split_into_sentences(text):
    """Split generated BHC into sentences for matching."""
    # Simple sentence splitter — split on period, exclamation, question mark
    # followed by space and capital letter, or end of string
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    # Filter out very short fragments
    return [s.strip() for s in sentences if len(s.strip()) > 15]


def compute_fact_recall(patient_id, generated_bhc, supported_props,
                        embed_model, threshold=0.65):
    """
    Measure what fraction of clinician-verified facts appear in a generated BHC.

    For each supported proposition, we find the most similar sentence in the
    generated BHC using cosine similarity. If similarity > threshold, we count
    it as "captured."

    Args:
        patient_id: The patient
        generated_bhc: The model's output text
        supported_props: DataFrame of supported propositions for this patient
        embed_model: BGE-M3 for encoding
        threshold: Cosine similarity threshold to count as a match

    Returns:
        dict with recall stats and per-proposition matches
    """
    # Get this patient's supported propositions
    patient_props = supported_props[supported_props['subject_id'] == patient_id]
    if len(patient_props) == 0:
        return None

    prop_texts = patient_props['text'].tolist()

    # Split generated BHC into sentences
    bhc_sentences = split_into_sentences(generated_bhc)
    if len(bhc_sentences) == 0:
        return {'patient_id': patient_id, 'n_props': len(prop_texts),
                'n_captured': 0, 'recall': 0.0, 'matches': []}

    # Embed both
    prop_embeddings = embed_model.encode(prop_texts).astype('float32')
    sent_embeddings = embed_model.encode(bhc_sentences).astype('float32')

    # Normalize for cosine similarity
    faiss.normalize_L2(prop_embeddings)
    faiss.normalize_L2(sent_embeddings)

    # For each proposition, find the best matching sentence
    # Using matrix multiplication for efficiency
    similarity_matrix = prop_embeddings @ sent_embeddings.T

    matches = []
    n_captured = 0
    for i, prop_text in enumerate(prop_texts):
        best_idx = similarity_matrix[i].argmax()
        best_score = similarity_matrix[i][best_idx]
        captured = best_score >= threshold
        if captured:
            n_captured += 1
        matches.append({
            'proposition': prop_text,
            'best_match': bhc_sentences[best_idx],
            'similarity': float(best_score),
            'captured': captured
        })

    recall = n_captured / len(prop_texts)

    return {
        'patient_id': patient_id,
        'n_props': len(prop_texts),
        'n_captured': n_captured,
        'recall': recall,
        'matches': matches
    }

In [36]:
# Test on patient 1084 to see if the matching works and calibrate threshold
test_result_bl = compute_fact_recall(
    1084,
    baseline_results[baseline_results['subject_id'] == 1084]['generated_bhc'].iloc[0],
    supported, embed_model, threshold=0.65
)

test_result_rag = compute_fact_recall(
    1084,
    rag2_results_df[rag2_results_df['subject_id'] == 1084]['generated_bhc'].iloc[0],
    supported, embed_model, threshold=0.65
)

print(f"Patient 1084 — Fact Recall")
print(f"  Supported propositions: {test_result_bl['n_props']}")
print(f"  Baseline: {test_result_bl['n_captured']}/{test_result_bl['n_props']} "
      f"({test_result_bl['recall']:.1%})")
print(f"  RAG v2:   {test_result_rag['n_captured']}/{test_result_rag['n_props']} "
      f"({test_result_rag['recall']:.1%})")

# Show some example matches and misses
print(f"\n--- EXAMPLE MATCHES (RAG v2) ---")
for m in sorted(test_result_rag['matches'], key=lambda x: x['similarity'], reverse=True)[:5]:
    status = "✓" if m['captured'] else "✗"
    print(f"\n  {status} Proposition: {m['proposition']}")
    print(f"    Best match ({m['similarity']:.3f}): {m['best_match'][:100]}...")

print(f"\n--- EXAMPLE MISSES (RAG v2) ---")
misses = [m for m in test_result_rag['matches'] if not m['captured']]
for m in sorted(misses, key=lambda x: x['similarity'], reverse=True)[:5]:
    print(f"\n  ✗ Proposition: {m['proposition']}")
    print(f"    Best match ({m['similarity']:.3f}): {m['best_match'][:100]}...")

Patient 1084 — Fact Recall
  Supported propositions: 57
  Baseline: 24/57 (42.1%)
  RAG v2:   28/57 (49.1%)

--- EXAMPLE MATCHES (RAG v2) ---

  ✓ Proposition: The patient was initially treated for a UTI as a possible source for his altered mental state.
    Best match (0.806): In the hospital, the patient was diagnosed with an altered mental status and a urinary tract infecti...

  ✓ Proposition: The patient was initially treated for a UTI as
a possible source for his altered mental state.  
    Best match (0.806): In the hospital, the patient was diagnosed with an altered mental status and a urinary tract infecti...

  ✓ Proposition: The patient was extubated.
    Best match (0.795): The patient was eventually discharged to the floor....

  ✓ Proposition: The patient has a history of prostate cancer.
    Best match (0.790): The patient is a 61-year-old male with a history of prostate cancer, pelvic osteoarthritis, hyperten...

  ✓ Proposition: He was treated with several medications 

In [37]:
# Run proposition-level evaluation on all 100 patients
# Compare baseline vs RAG v2

print("Running proposition-level fact recall on all 100 patients...")
print("(Embedding propositions + sentences for 200 BHCs — a few minutes)")
print("-" * 60)

start = time.time()

baseline_fact_results = []
rag2_fact_results = []

for i, pid in enumerate(human_bhcs['subject_id']):
    # Baseline
    bl_bhc = baseline_results[baseline_results['subject_id'] == pid]['generated_bhc'].iloc[0]
    bl_result = compute_fact_recall(pid, bl_bhc, supported, embed_model, threshold=0.60)
    if bl_result:
        baseline_fact_results.append(bl_result)

    # RAG v2
    rag_bhc = rag2_results_df[rag2_results_df['subject_id'] == pid]['generated_bhc'].iloc[0]
    rag_result = compute_fact_recall(pid, rag_bhc, supported, embed_model, threshold=0.60)
    if rag_result:
        rag2_fact_results.append(rag_result)

    if (i + 1) % 20 == 0:
        print(f"  {i+1}/100 patients complete")

elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s")

# Convert to DataFrames
bl_fact_df = pd.DataFrame([{
    'subject_id': r['patient_id'],
    'n_props': r['n_props'],
    'n_captured': r['n_captured'],
    'fact_recall': r['recall']
} for r in baseline_fact_results])

rag2_fact_df = pd.DataFrame([{
    'subject_id': r['patient_id'],
    'n_props': r['n_props'],
    'n_captured': r['n_captured'],
    'fact_recall': r['recall']
} for r in rag2_fact_results])

# Merge for comparison
fact_comparison = bl_fact_df.rename(
    columns={'n_captured': 'bl_captured', 'fact_recall': 'bl_recall'}
).merge(
    rag2_fact_df.rename(
        columns={'n_captured': 'rag_captured', 'fact_recall': 'rag_recall'}
    )[['subject_id', 'rag_captured', 'rag_recall']],
    on='subject_id'
)

# Add truncation status
fact_comparison = fact_comparison.merge(
    baseline_results[['subject_id', 'was_truncated']],
    on='subject_id'
)

fact_comparison['delta_recall'] = fact_comparison['rag_recall'] - fact_comparison['bl_recall']

Running proposition-level fact recall on all 100 patients...
(Embedding propositions + sentences for 200 BHCs — a few minutes)
------------------------------------------------------------
  20/100 patients complete
  40/100 patients complete
  60/100 patients complete
  80/100 patients complete
  100/100 patients complete
Done in 32.3s


In [38]:
# PROPOSITION-LEVEL RESULTS
print("=" * 70)
print("PROPOSITION-LEVEL FACT RECALL (threshold=0.60)")
print("=" * 70)

print(f"\nFULL COHORT (n={len(fact_comparison)})")
print(f"  Total verified propositions: {fact_comparison['n_props'].sum()}")
print(f"  {'Metric':<24} {'Baseline':<16} {'RAG v2':<16} {'Δ'}")
print(f"  {'-'*56}")
print(f"  {'Mean fact recall':<24} {fact_comparison['bl_recall'].mean():.1%}           "
      f"{fact_comparison['rag_recall'].mean():.1%}           "
      f"{fact_comparison['delta_recall'].mean():+.1%}")
print(f"  {'Median fact recall':<24} {fact_comparison['bl_recall'].median():.1%}           "
      f"{fact_comparison['rag_recall'].median():.1%}")
print(f"  {'Total facts captured':<24} {fact_comparison['bl_captured'].sum()}/{fact_comparison['n_props'].sum()}        "
      f"{fact_comparison['rag_captured'].sum()}/{fact_comparison['n_props'].sum()}")

# Split by truncation
trunc_facts = fact_comparison[fact_comparison['was_truncated'] == True]
no_trunc_facts = fact_comparison[fact_comparison['was_truncated'] == False]

print(f"\nPREVIOUSLY TRUNCATED (n={len(trunc_facts)})")
print(f"  {'Mean fact recall':<24} {trunc_facts['bl_recall'].mean():.1%}           "
      f"{trunc_facts['rag_recall'].mean():.1%}           "
      f"{trunc_facts['delta_recall'].mean():+.1%}")
print(f"  {'Total facts captured':<24} {trunc_facts['bl_captured'].sum()}/{trunc_facts['n_props'].sum()}        "
      f"{trunc_facts['rag_captured'].sum()}/{trunc_facts['n_props'].sum()}")

print(f"\nNOT TRUNCATED (n={len(no_trunc_facts)})")
print(f"  {'Mean fact recall':<24} {no_trunc_facts['bl_recall'].mean():.1%}           "
      f"{no_trunc_facts['rag_recall'].mean():.1%}           "
      f"{no_trunc_facts['delta_recall'].mean():+.1%}")
print(f"  {'Total facts captured':<24} {no_trunc_facts['bl_captured'].sum()}/{no_trunc_facts['n_props'].sum()}        "
      f"{no_trunc_facts['rag_captured'].sum()}/{no_trunc_facts['n_props'].sum()}")

# Win/loss
print(f"\nPER-PATIENT WINS/LOSSES (fact recall):")
print(f"  Truncated:     {(trunc_facts['delta_recall'] > 0).sum()}/{len(trunc_facts)} improved, "
      f"{(trunc_facts['delta_recall'] < 0).sum()}/{len(trunc_facts)} worsened, "
      f"{(trunc_facts['delta_recall'] == 0).sum()}/{len(trunc_facts)} tied")
print(f"  Not truncated: {(no_trunc_facts['delta_recall'] > 0).sum()}/{len(no_trunc_facts)} improved, "
      f"{(no_trunc_facts['delta_recall'] < 0).sum()}/{len(no_trunc_facts)} worsened, "
      f"{(no_trunc_facts['delta_recall'] == 0).sum()}/{len(no_trunc_facts)} tied")

# Save
fact_comparison.to_parquet(f"{DATA_DIR}/proposition_eval_results.parquet", index=False)
print(f"\nSaved proposition evaluation to Drive")

PROPOSITION-LEVEL FACT RECALL (threshold=0.60)

FULL COHORT (n=100)
  Total verified propositions: 5396
  Metric                   Baseline         RAG v2           Δ
  --------------------------------------------------------
  Mean fact recall         73.5%           72.2%           -1.3%
  Median fact recall       76.0%           75.3%
  Total facts captured     3735/5396        3662/5396

PREVIOUSLY TRUNCATED (n=42)
  Mean fact recall         70.0%           68.3%           -1.7%
  Total facts captured     1849/2819        1759/2819

NOT TRUNCATED (n=58)
  Mean fact recall         76.0%           75.0%           -1.1%
  Total facts captured     1886/2577        1903/2577

PER-PATIENT WINS/LOSSES (fact recall):
  Truncated:     15/42 improved, 26/42 worsened, 1/42 tied
  Not truncated: 19/58 improved, 29/58 worsened, 10/58 tied

Saved proposition evaluation to Drive


In [39]:
# Let's understand WHY baseline is holding up so well
# even for truncated patients

# How much content does baseline actually keep for truncated patients?
trunc_patients = baseline_results[baseline_results['was_truncated'] == True]

print("WHAT BASELINE ACTUALLY DOES FOR TRUNCATED PATIENTS:")
print("=" * 60)
print(f"Truncation strategy: keep FIRST 31K tokens, discard the rest")
print(f"Truncated patients: {len(trunc_patients)}")
print(f"\nFull note tokens (before truncation):")
print(f"  Mean:   {trunc_patients['input_tokens'].mean():.0f}")
print(f"  Median: {trunc_patients['input_tokens'].median():.0f}")
print(f"\nBaseline keeps the first 31K tokens — which means it keeps:")
print(f"  - Admission notes (earliest, most important)")
print(f"  - ED course and initial workup")
print(f"  - Early physician assessments and plans")
print(f"  - Early nursing documentation")
print(f"\nBaseline LOSES (truncated from end):")
print(f"  - Later progress notes (often repetitive)")
print(f"  - Later nursing notes (vital sign documentation)")
print(f"  - Discharge planning notes")

# What does RAG select instead?
print(f"\n\nWHAT RAG RETRIEVES:")
print(f"  Retrieved tokens: mean {rag2_results_df['retrieved_tokens'].mean():.0f}")
print(f"  This is LESS context than baseline's 31K truncated input")

# This is the key insight
print(f"\n{'='*60}")
print(f"KEY INSIGHT: RAG gives the model LESS total context")
print(f"  Baseline (truncated): ~31,000 tokens of continuous narrative")
print(f"  RAG v2:               ~{rag2_results_df['retrieved_tokens'].mean():.0f} tokens of selected chunks")
print(f"  Difference:           ~{31000 - rag2_results_df['retrieved_tokens'].mean():.0f} fewer tokens with RAG")
print(f"{'='*60}")

WHAT BASELINE ACTUALLY DOES FOR TRUNCATED PATIENTS:
Truncation strategy: keep FIRST 31K tokens, discard the rest
Truncated patients: 42

Full note tokens (before truncation):
  Mean:   105563
  Median: 63602

Baseline keeps the first 31K tokens — which means it keeps:
  - Admission notes (earliest, most important)
  - ED course and initial workup
  - Early physician assessments and plans
  - Early nursing documentation

Baseline LOSES (truncated from end):
  - Later progress notes (often repetitive)
  - Later nursing notes (vital sign documentation)
  - Discharge planning notes


WHAT RAG RETRIEVES:
  Retrieved tokens: mean 10273
  This is LESS context than baseline's 31K truncated input

KEY INSIGHT: RAG gives the model LESS total context
  Baseline (truncated): ~31,000 tokens of continuous narrative
  RAG v2:               ~10273 tokens of selected chunks
  Difference:           ~20727 fewer tokens with RAG


In [40]:
# RAG v3: Match baseline's token budget
# Baseline truncated patients get ~31K tokens
# Let's give RAG the same budget and see if more selected content helps

def retrieve_multi_query_v3(patient_id, queries, embed_model, patient_indices,
                            patient_chunks, chunks_per_query=20, max_total_tokens=28000):
    """
    Same multi-query approach but with a much larger token budget
    to match baseline's ~31K context window.
    """
    all_selected = []
    seen_fingerprints = set()

    for section, query in queries.items():
        query_emb = embed_model.encode([query]).astype('float32')
        faiss.normalize_L2(query_emb)

        p_chunks = patient_chunks[patient_id]
        actual_fetch = min(chunks_per_query * 3, patient_indices[patient_id].ntotal)
        scores, indices = patient_indices[patient_id].search(query_emb, actual_fetch)

        section_count = 0
        for idx, score in zip(indices[0], scores[0]):
            if section_count >= chunks_per_query:
                break

            chunk = p_chunks.iloc[idx]
            text = chunk['chunk_text']
            mid = len(text) // 2
            fingerprint = text[max(0, mid-100):mid+100]

            is_dup = False
            for seen in seen_fingerprints:
                overlap = len(set(fingerprint.split()) & set(seen.split()))
                total = max(len(set(fingerprint.split())), 1)
                if overlap / total > 0.6:
                    is_dup = True
                    break

            if not is_dup:
                all_selected.append({
                    **chunk.to_dict(),
                    'similarity_score': score,
                    'query_section': section
                })
                seen_fingerprints.add(fingerprint)
                section_count += 1

    retrieved = pd.DataFrame(all_selected)
    retrieved = retrieved.sort_values('similarity_score', ascending=False)
    cumulative_tokens = retrieved['chunk_tokens'].cumsum()
    retrieved = retrieved[cumulative_tokens <= max_total_tokens]
    retrieved = retrieved.sort_values('chartdate')
    total_tokens = retrieved['chunk_tokens'].sum()

    return retrieved, total_tokens


# Quick test — how much context does this give us?
test_ret, test_tok = retrieve_multi_query_v3(
    test_pid, BHC_QUERIES, embed_model, patient_indices, patient_chunks
)
print(f"Patient {test_pid}: {len(test_ret)} chunks, {test_tok} tokens")
print(f"  (vs baseline: 31K tokens, vs RAG v2: ~10K tokens)")
print(f"\nChunks per query section:")
print(test_ret['query_section'].value_counts().to_string())

Patient 1084: 44 chunks, 11813 tokens
  (vs baseline: 31K tokens, vs RAG v2: ~10K tokens)

Chunks per query section:
query_section
admission    20
diagnoses    19
treatment     3
workup        2


In [41]:
# Test on a heavily truncated patient where the budget difference should matter
large_patient = baseline_results[baseline_results['was_truncated'] == True].sort_values(
    'input_tokens', ascending=False
).iloc[0]['subject_id']

large_tokens = notes[notes['SUBJECT_ID'] == large_patient]['TEXT'].apply(
    lambda x: len(tokenizer.encode(x))
).sum()

# RAG v2 (~10K budget)
ret_v2, tok_v2 = retrieve_multi_query(
    large_patient, BHC_QUERIES, embed_model, patient_indices, patient_chunks
)

# RAG v3 (~28K budget)
ret_v3, tok_v3 = retrieve_multi_query_v3(
    large_patient, BHC_QUERIES, embed_model, patient_indices, patient_chunks
)

print(f"Patient {large_patient} — {large_tokens} total tokens")
print(f"  Baseline sees:  ~31,000 tokens (first 31K, rest discarded)")
print(f"  RAG v2 selects: {tok_v2} tokens ({len(ret_v2)} chunks)")
print(f"  RAG v3 selects: {tok_v3} tokens ({len(ret_v3)} chunks)")
print(f"\nRAG v3 query section breakdown:")
print(ret_v3['query_section'].value_counts().to_string())
print(f"\nRAG v3 note category breakdown:")
print(ret_v3['category'].value_counts().to_string())

Patient 93640 — 356448 total tokens
  Baseline sees:  ~31,000 tokens (first 31K, rest discarded)
  RAG v2 selects: 11637 tokens (50 chunks)
  RAG v3 selects: 27712 tokens (109 chunks)

RAG v3 query section breakdown:
query_section
admission    20
discharge    20
treatment    20
diagnoses    17
course       16
workup       16

RAG v3 note category breakdown:
category
Physician         73
Rehab Services    15
Nursing           10
Radiology          5
Respiratory        5
General            1


In [42]:
# Run RAG v3 (matched token budget) on all 100 patients
rag3_results = []

print(f"Running RAG v3 (28K token budget) on {len(human_bhcs)} patients...")
print("-" * 60)

start_total = time.time()

for i, row in human_bhcs.iterrows():
    patient_id = row['subject_id']

    retrieved_df, retrieved_tokens = retrieve_multi_query_v3(
        patient_id, BHC_QUERIES, embed_model, patient_indices, patient_chunks
    )

    start = time.time()
    generated, input_tokens = generate_bhc_rag(retrieved_df, model, tokenizer)
    elapsed = time.time() - start

    full_tokens = notes[notes['SUBJECT_ID'] == patient_id]['TEXT'].apply(
        lambda x: len(tokenizer.encode(x))
    ).sum()

    was_truncated_baseline = baseline_results[
        baseline_results['subject_id'] == patient_id
    ]['was_truncated'].iloc[0]

    rag3_results.append({
        'subject_id': patient_id,
        'full_note_tokens': full_tokens,
        'retrieved_tokens': retrieved_tokens,
        'input_tokens': input_tokens,
        'compression_ratio': retrieved_tokens / full_tokens,
        'was_truncated_baseline': was_truncated_baseline,
        'generated_bhc': generated,
        'generated_tokens': len(tokenizer.encode(generated)),
        'generation_time_sec': round(elapsed, 1),
        'human_bhc': row['brief_hospital_course'],
        'n_chunks_retrieved': len(retrieved_df)
    })

    if len(rag3_results) % 10 == 0:
        print(f"  {len(rag3_results)}/100 | Last: {full_tokens} full → "
              f"{retrieved_tokens} retrieved ({retrieved_tokens/full_tokens:.0%}) | "
              f"{elapsed:.1f}s | Baseline truncated: {was_truncated_baseline}")

total_time = time.time() - start_total
print("-" * 60)
print(f"Done! Total time: {total_time/60:.1f} minutes")

rag3_results_df = pd.DataFrame(rag3_results)
rag3_results_df.to_parquet(f"{DATA_DIR}/rag3_matched_budget_results.parquet", index=False)
print(f"Saved {len(rag3_results_df)} results to Drive")

Running RAG v3 (28K token budget) on 100 patients...
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  10/100 | Last: 95397 full → 25665 retrieved (27%) | 18.0s | Baseline truncated: True


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  20/100 | Last: 195180 full → 27923 retrieved (14%) | 23.3s | Baseline truncated: True


...

  80/100 | Last: 23144 full → 17081 retrieved (74%) | 31.6s | Baseline truncated: False


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  90/100 | Last: 22100 full → 19236 retrieved (87%) | 19.8s | Baseline truncated: False


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  100/100 | Last: 26950 full → 14605 retrieved (54%) | 26.0s | Baseline truncated: False
------------------------------------------------------------
Done! Total time: 62.8 minutes
Saved 100 results to Drive


In [43]:
# ROUGE scores
rag3_rouge = []
for _, row in rag3_results_df.iterrows():
    scores = scorer.score(row['human_bhc'], row['generated_bhc'])
    rag3_rouge.append({
        'subject_id': row['subject_id'],
        'rouge1_f': scores['rouge1'].fmeasure,
        'rouge2_f': scores['rouge2'].fmeasure,
        'rougeL_f': scores['rougeL'].fmeasure,
    })
rag3_rouge_df = pd.DataFrame(rag3_rouge)

# BERTScore
print("Computing BERTScore...")
P, R, F1 = bert_score_fn(
    rag3_results_df['generated_bhc'].tolist(),
    rag3_results_df['human_bhc'].tolist(),
    lang='en',
    verbose=True
)
rag3_results_df['bertscore_f1'] = F1.numpy()

# Proposition-level fact recall
print("\nComputing proposition-level fact recall...")
rag3_fact_results = []
for i, pid in enumerate(human_bhcs['subject_id']):
    rag_bhc = rag3_results_df[rag3_results_df['subject_id'] == pid]['generated_bhc'].iloc[0]
    result = compute_fact_recall(pid, rag_bhc, supported, embed_model, threshold=0.60)
    if result:
        rag3_fact_results.append(result)
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/100 patients complete")

rag3_fact_df = pd.DataFrame([{
    'subject_id': r['patient_id'],
    'n_props': r['n_props'],
    'n_captured': r['n_captured'],
    'fact_recall': r['recall']
} for r in rag3_fact_results])

# Save everything
rag3_eval = rag3_results_df.merge(rag3_rouge_df, on='subject_id')
rag3_eval.to_parquet(f"{DATA_DIR}/rag3_matched_budget_eval_results.parquet", index=False)
print("Saved RAG v3 evaluation results to Drive")

Computing BERTScore...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 3.13 seconds, 31.94 sentences/sec

Computing proposition-level fact recall...
  20/100 patients complete
  40/100 patients complete
  60/100 patients complete
  80/100 patients complete
  100/100 patients complete
Saved RAG v3 evaluation results to Drive


In [44]:
# ================================================================
# DEFINITIVE FOUR-WAY COMPARISON
# ================================================================

print("=" * 80)
print("DEFINITIVE COMPARISON — Baseline vs All RAG Variants")
print("=" * 80)

print(f"\n{'AUTOMATED METRICS (n=100)':}")
print(f"  {'Metric':<16} {'Baseline':<15} {'RAG v1':<15} {'RAG v2':<15} {'RAG v3':<15}")
print(f"  {'':.<16} {'(zero-shot)':<15} {'(single-Q)':<15} {'(multi-Q)':<15} {'(multi-Q 28K)':<15}")
print(f"  {'-'*76}")

for name, bl_col, r1_col, r2_col in [
    ('ROUGE-1 F1', 'rouge1_f', 'rouge1_f', 'rouge1_f'),
    ('ROUGE-2 F1', 'rouge2_f', 'rouge2_f', 'rouge2_f'),
    ('ROUGE-L F1', 'rougeL_f', 'rougeL_f', 'rougeL_f'),
]:
    bl = baseline_eval[bl_col].mean()
    r1 = rag_eval[r1_col].mean()
    r2 = rag2_eval[r2_col].mean()
    r3 = rag3_rouge_df[r2_col].mean()
    print(f"  {name:<16} {bl:.3f}          {r1:.3f}          {r2:.3f}          {r3:.3f}")

bl_b = baseline_eval['bertscore_f1'].mean()
r1_b = rag_eval['bertscore_f1'].mean()
r2_b = rag2_eval['bertscore_f1'].mean()
r3_b = rag3_results_df['bertscore_f1'].mean()
print(f"  {'BERTScore F1':<16} {bl_b:.3f}          {r1_b:.3f}          {r2_b:.3f}          {r3_b:.3f}")

print(f"\n  {'Context size':<16} {'~31K (trunc)':<15} {'~11K':<15} {'~10K':<15} {'~{:.0f}K'.format(rag3_results_df['retrieved_tokens'].mean()/1000):<15}")

# Proposition-level
print(f"\n{'PROPOSITION-LEVEL FACT RECALL':}")
print(f"  {'Metric':<24} {'Baseline':<16} {'RAG v2':<16} {'RAG v3':<16}")
print(f"  {'-'*64}")
print(f"  {'Mean fact recall':<24} {fact_comparison['bl_recall'].mean():.1%}           "
      f"{fact_comparison['rag_recall'].mean():.1%}           "
      f"{rag3_fact_df['fact_recall'].mean():.1%}")
print(f"  {'Facts captured':<24} "
      f"{fact_comparison['bl_captured'].sum()}/{fact_comparison['n_props'].sum()}        "
      f"{fact_comparison['rag_captured'].sum()}/{fact_comparison['n_props'].sum()}        "
      f"{rag3_fact_df['n_captured'].sum()}/{rag3_fact_df['n_props'].sum()}")

# Subgroup for v3
v3_fact_comp = fact_comparison[['subject_id', 'n_props', 'bl_captured', 'bl_recall', 'was_truncated']].merge(
    rag3_fact_df[['subject_id', 'n_captured', 'fact_recall']].rename(
        columns={'n_captured': 'v3_captured', 'fact_recall': 'v3_recall'}
    ),
    on='subject_id'
)
v3_fact_comp['delta_recall'] = v3_fact_comp['v3_recall'] - v3_fact_comp['bl_recall']

trunc_v3 = v3_fact_comp[v3_fact_comp['was_truncated'] == True]
no_trunc_v3 = v3_fact_comp[v3_fact_comp['was_truncated'] == False]

print(f"\n{'FACT RECALL BY SUBGROUP':}")
print(f"  {'Group':<28} {'Baseline':<16} {'RAG v3':<16} {'Δ'}")
print(f"  {'-'*60}")
print(f"  {'Truncated (n=42)':<28} {trunc_v3['bl_recall'].mean():.1%}           "
      f"{trunc_v3['v3_recall'].mean():.1%}           "
      f"{trunc_v3['delta_recall'].mean():+.1%}")
print(f"  {'Not truncated (n=58)':<28} {no_trunc_v3['bl_recall'].mean():.1%}           "
      f"{no_trunc_v3['v3_recall'].mean():.1%}           "
      f"{no_trunc_v3['delta_recall'].mean():+.1%}")

print(f"\n{'PER-PATIENT WINS/LOSSES (fact recall, RAG v3 vs baseline)':}")
print(f"  Truncated:     {(trunc_v3['delta_recall'] > 0).sum()}/42 improved, "
      f"{(trunc_v3['delta_recall'] < 0).sum()}/42 worsened, "
      f"{(trunc_v3['delta_recall'] == 0).sum()}/42 tied")
print(f"  Not truncated: {(no_trunc_v3['delta_recall'] > 0).sum()}/58 improved, "
      f"{(no_trunc_v3['delta_recall'] < 0).sum()}/58 worsened, "
      f"{(no_trunc_v3['delta_recall'] == 0).sum()}/58 tied")

DEFINITIVE COMPARISON — Baseline vs All RAG Variants

AUTOMATED METRICS (n=100)
  Metric           Baseline        RAG v1          RAG v2          RAG v3         
  ................ (zero-shot)     (single-Q)      (multi-Q)       (multi-Q 28K)  
  ----------------------------------------------------------------------------
  ROUGE-1 F1       0.339          0.339          0.335          0.334
  ROUGE-2 F1       0.084          0.079          0.081          0.084
  ROUGE-L F1       0.165          0.163          0.162          0.168
  BERTScore F1     0.819          0.818          0.818          0.820

  Context size     ~31K (trunc)    ~11K            ~10K            ~18K           

PROPOSITION-LEVEL FACT RECALL
  Metric                   Baseline         RAG v2           RAG v3          
  ----------------------------------------------------------------
  Mean fact recall         73.5%           72.2%           72.5%
  Facts captured           3735/5396        3662/5396        3711/5396

In [45]:
# Save all v3 fact recall results
v3_fact_comp.to_parquet(f"{DATA_DIR}/rag3_proposition_eval.parquet", index=False)

# Print all saved files
print("ALL RESULTS SAVED TO DRIVE:")
print("-" * 50)
for f in sorted(os.listdir(DATA_DIR)):
    size = os.path.getsize(f"{DATA_DIR}/{f}") / 1024
    print(f"  {f:<50} {size:.0f} KB")

ALL RESULTS SAVED TO DRIVE:
--------------------------------------------------
  baseline_eval_results.parquet                      243 KB
  baseline_results.parquet                           235 KB
  human_bhcs.parquet                                 146 KB
  notes.parquet                                      5241 KB
  proposition_eval_results.parquet                   9 KB
  propositions_with_gt.parquet                       1310 KB
  rag2_multi_query_eval_results.parquet              252 KB
  rag2_multi_query_results.parquet                   245 KB
  rag3_matched_budget_eval_results.parquet           244 KB
  rag3_matched_budget_results.parquet                238 KB
  rag3_proposition_eval.parquet                      9 KB
  rag_eval_results.parquet                           251 KB
  rag_results.parquet                                243 KB
